# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
#%load_ext dotenv
#%dotenv ../05_src/.secrets

import os
from dotenv import load_dotenv

load_dotenv("../05_src/.secrets", override=True)

print(repr(os.getenv("OPENAI_API_KEY")))

'any_value'


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from langchain_community.document_loaders import PyPDFLoader

#file_path = "../docs/Managing_Oneself_Drucker_HBR.pdf" # not from local file, but from URL.

url = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"

loader = PyPDFLoader(url)

docs = loader.load()

print(len(docs))

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

#print(document_text) #I am not printing the whole document text here, but you can see it in your environment.


13


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
from openai import OpenAI
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})



#user specified tone. this could be modified to ask the user for the tone in the program

tone  = "Legalese"

user_prompt = f"""
    
    Given the text from the article downloaded from the pdf, do the following:
    
    1. Identify the book's author
    2. Identify the book's title
    3. Identify Relevance by creating a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    4. Create a Summary that gives a concise and succinct summary of the text no longer than 1000 tokens. The summary should use the tone stored here: {tone}

    The text from the pdf is located here: 
    
    <article>
    {document_text}
    </article>

    
    Provide your response in the following format:
    Title: <title>
    Author: <author>
    Relevance: <relevance>
    Summary: <summary>
"""

#system prompt

system_prompt = "You are new to AI and you are trying to understand why this article is relevant for an AI professional in their professional development."

response = client.responses.create(
    #model = 'gpt-4o',
    model = 'gpt-4o-mini', # depending on the tier we have available, we might need to update the model to be used
    instructions = system_prompt,
    input = user_prompt,
)

from IPython.display import display, Markdown

display(Markdown(response.output_text))

#display the specified tone and the input/output tokens based on the response object
print(f"Tone: {tone}")
print(f"InputTokens: {response.usage.input_tokens}")
print(f"OutputTokens: {response.usage.output_tokens}")

**Title:** Managing Oneself  
**Author:** Peter F. Drucker  
**Relevance:** This article is paramount for AI professionals' career development as it emphasizes self-awareness, understanding personal strengths, and aligning values with professional contributions. In a rapidly evolving field like AI, where individuals must continuously adapt and pivot, fostering a keen self-understanding enables professionals to navigate their careers effectively and make meaningful impacts within their organizations.  

**Summary:**  
This text elucidates the necessity of self-management in contemporary knowledge work. As companies no longer oversee their employees' career paths, individuals must become their own chief executive officers. Drucker posits that success hinges on self-knowledge, prompting professionals to introspectively evaluate their strengths, learning modalities, values, and optimal work environments. He advocates for feedback analysis to accurately discern personal strengths, encouraging individuals to concentrate on and amplify these attributes while avoiding areas of incompetence. The discourse extends to recognizing diverse performance styles and learning preferences, with Drucker underscoring that self-awareness in these dimensions is crucial for achieving results in collaborative settings. Furthermore, he navigates the significance of aligning one’s values with those of the organizations they engage with to circumvent frustration and enhance effectiveness. The text concludes with the imperative of building and nurturing relationships within work environments, necessitating clear communication about individual contributions and expectations. Drucker envisions that managing oneself is not merely individualistic but an essential capacity for thriving amidst the organization-centric expectations of the knowledge economy. The call to action is explicit: knowledge workers must actively engage in self-reflection and management to realize their potential and to contribute meaningfully in their professional landscapes.

Tone: Legalese
InputTokens: 12337
OutputTokens: 328


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [10]:
import re

output_text = response.output_text

title_match = re.search(r"\*\*Title:\*\*\s*(.*)", output_text)
author_match = re.search(r"\*\*Author:\*\*\s*(.*)", output_text)
relevance_match = re.search(
    r"\*\*Relevance:\*\*\s*(.*?)(?=\n\s*\*\*Summary:\*\*)",
    output_text,
    re.DOTALL
)
summary_match = re.search(
    r"\*\*Summary:\*\*\s*(.*)",
    output_text,
    re.DOTALL
)

title = title_match.group(1).strip() if title_match else ""
author = author_match.group(1).strip() if author_match else ""
relevance = relevance_match.group(1).strip() if relevance_match else ""
summary_text = summary_match.group(1).strip() if summary_match else ""

print("TITLE:", title)
print("AUTHOR:", author)
print("RELEVANCE:", relevance)
print("SUMMARY:", summary_text)

TITLE: Managing Oneself
AUTHOR: Peter F. Drucker
RELEVANCE: This article is paramount for AI professionals' career development as it emphasizes self-awareness, understanding personal strengths, and aligning values with professional contributions. In a rapidly evolving field like AI, where individuals must continuously adapt and pivot, fostering a keen self-understanding enables professionals to navigate their careers effectively and make meaningful impacts within their organizations.
SUMMARY: This text elucidates the necessity of self-management in contemporary knowledge work. As companies no longer oversee their employees' career paths, individuals must become their own chief executive officers. Drucker posits that success hinges on self-knowledge, prompting professionals to introspectively evaluate their strengths, learning modalities, values, and optimal work environments. He advocates for feedback analysis to accurately discern personal strengths, encouraging individuals to concent

In [11]:
from deepeval.models import GPTModel

eval_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
)


In [12]:
#Deepeval libs
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [13]:
#Creating a test case for the summary evaluation using the extracted summary_text as the actual output and the original document_text as the input.
test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_text
)


In [16]:
print("----- FULL RAW OUTPUT -----")
print(output_text)

print("\n----- EXTRACTED SUMMARY -----")
print(summary_text)

----- FULL RAW OUTPUT -----
**Title:** Managing Oneself  
**Author:** Peter F. Drucker  
**Relevance:** This article is paramount for AI professionals' career development as it emphasizes self-awareness, understanding personal strengths, and aligning values with professional contributions. In a rapidly evolving field like AI, where individuals must continuously adapt and pivot, fostering a keen self-understanding enables professionals to navigate their careers effectively and make meaningful impacts within their organizations.  

**Summary:**  
This text elucidates the necessity of self-management in contemporary knowledge work. As companies no longer oversee their employees' career paths, individuals must become their own chief executive officers. Drucker posits that success hinges on self-knowledge, prompting professionals to introspectively evaluate their strengths, learning modalities, values, and optimal work environments. He advocates for feedback analysis to accurately discern p

In [14]:
#Summarization metric with 5 bespoke questions that are relevant for evaluating the quality of a summary, along with a threshold of 0.5 to determine if the summary is acceptable or not based on the average score across these questions.
summarization_questions = [
    "Does the summary accurately capture the central idea of the article?",
    "Does the summary include the most important supporting arguments and ideas?",
    "Does the summary avoid introducing claims that are not supported by the article?",
    "Does the summary preserve the meaning and intent of the original article?",
    "Is the summary concise while still covering the essential points?"
]

summarization_metric = SummarizationMetric(
    assessment_questions=summarization_questions,
    threshold=0.5,
    model=eval_model,
)

summarization_metric.measure(test_case)

print("Summarization score:", summarization_metric.score)
print("Summarization reason:", summarization_metric.reason)


Output()

Summarization score: 0.8571428571428571
Summarization reason: The score is 0.86 because the summary captures the main ideas of the original text effectively, despite including extra information about recognizing diverse performance styles and clear communication that was not present in the original text.


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [18]:
from pydantic import BaseModel

class EvaluationOutput(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

def evaluate_summary(document_text, summary_text, tone, eval_model):
    from deepeval.metrics import SummarizationMetric, GEval
    from deepeval.test_case import LLMTestCase, LLMTestCaseParams

    test_case = LLMTestCase(
        input=document_text,
        actual_output=summary_text
    )

    summarization_questions = [
        "Does the summary accurately capture Drucker's main argument about self-management?",
        "Does the summary preserve the article's key ideas about strengths, performance, values, and contribution?",
        "Does the summary avoid adding claims not explicitly supported by the article?",
        "Does the summary preserve the intended meaning of the original article?",
        "Is the summary concise while still covering the essential points?"
    ]

    summarization_metric = SummarizationMetric(
        assessment_questions=summarization_questions,
        threshold=0.5,
        model=eval_model,
    )
    summarization_metric.measure(test_case)

    coherence_metric = GEval(
        name="Coherence",
        evaluation_steps=[
            "Check whether the summary follows a logical order.",
            "Check whether the ideas connect clearly from sentence to sentence.",
            "Check whether the writing is easy to understand.",
            "Check whether the summary avoids contradictions or confusing statements.",
            "Check whether the overall structure is clear and organized."
        ],
        evaluation_params=[
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT
        ],
        model=eval_model,
    )
    coherence_metric.measure(test_case)

    tonality_metric = GEval(
        name="Tonality",
        evaluation_steps=[
            f"Check whether the summary reflects the requested tone: {tone}.",
            "Check whether the tone is consistent throughout the summary.",
            "Check whether the tone is clearly distinguishable.",
            "Check whether the tone supports readability.",
            "Check whether the style matches the requested writing approach."
        ],
        evaluation_params=[
            LLMTestCaseParams.ACTUAL_OUTPUT
        ],
        model=eval_model,
    )
    tonality_metric.measure(test_case)

    safety_metric = GEval(
        name="Safety",
        evaluation_steps=[
            "Check whether the summary avoids harmful or offensive language.",
            "Check whether the summary avoids fabricated claims.",
            "Check whether the summary does not misrepresent the article in a misleading way.",
            "Check whether the summary is appropriate for an academic setting.",
            "Check whether the summary avoids unsafe or inappropriate content."
        ],
        evaluation_params=[
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT
        ],
        model=eval_model,
    )
    safety_metric.measure(test_case)

    return EvaluationOutput(
        SummarizationScore=summarization_metric.score,
        SummarizationReason=summarization_metric.reason,
        CoherenceScore=coherence_metric.score,
        CoherenceReason=coherence_metric.reason,
        TonalityScore=tonality_metric.score,
        TonalityReason=tonality_metric.reason,
        SafetyScore=safety_metric.score,
        SafetyReason=safety_metric.reason
    )

In [19]:
initial_evaluation = evaluate_summary(
    document_text=document_text,
    summary_text=summary_text,
    tone=tone,
    eval_model=eval_model
)

print(initial_evaluation.model_dump())

Output()

Output()

Output()

Output()

{'SummarizationScore': 0.8571428571428571, 'SummarizationReason': 'The score is 0.86 because the summary captures the main ideas of the original text effectively, despite including some extra information that was not present in the original. This additional context may enhance understanding but does not detract from the overall accuracy.', 'CoherenceScore': 0.8731058572770513, 'CoherenceReason': 'The response follows a logical order, clearly outlining the key concepts of self-management as presented by Drucker. Ideas connect well from sentence to sentence, making the writing easy to understand. The summary avoids contradictions and maintains clarity throughout. The overall structure is organized, effectively summarizing the main points of the original text while capturing its essence. However, a slight improvement could be made in explicitly linking some ideas to the specific questions Drucker poses, which would enhance the connection to the original content.', 'TonalityScore': 0.25361

In [21]:
#Bulding an enhancement prompt to improve the summary based on the evaluation results, asking the model to specifically address the weaknesses identified in the evaluation while maintaining the requested tone and ensuring that the summary is concise and accurate.

enhancement_system_prompt = """
You are a careful editor.
Revise summaries using evaluation feedback.
Your goal is to improve factual faithfulness, coherence, tone consistency, and safety.
Do not add new information not supported by the source article.
"""

enhancement_user_prompt = f"""
You will improve a summary of an article using the original article and evaluation feedback.

Requested tone:
{tone}

Original article:
<article>
{document_text}
</article>

Current summary:
<current_summary>
{summary_text}
</current_summary>

Evaluation feedback:
<summarization_score>{initial_evaluation.SummarizationScore}</summarization_score>
<summarization_reason>{initial_evaluation.SummarizationReason}</summarization_reason>

<coherence_score>{initial_evaluation.CoherenceScore}</coherence_score>
<coherence_reason>{initial_evaluation.CoherenceReason}</coherence_reason>

<tonality_score>{initial_evaluation.TonalityScore}</tonality_score>
<tonality_reason>{initial_evaluation.TonalityReason}</tonality_reason>

<safety_score>{initial_evaluation.SafetyScore}</safety_score>
<safety_reason>{initial_evaluation.SafetyReason}</safety_reason>

Instructions:
1. Rewrite the summary so it is more faithful to the article.
2. Fix any issues mentioned in the evaluation reasons.
3. Keep the summary concise.
4. Do not include title, author, or relevance.
5. Return only the improved summary text.
"""


In [22]:
#Enhanced summary

enhanced_response = client.responses.create(
    model="gpt-4o-mini",
    instructions=enhancement_system_prompt,
    input=enhancement_user_prompt,
    temperature=0
)

enhanced_summary_text = enhanced_response.output_text.strip()

print(enhanced_summary_text)

This text elucidates the necessity of self-management in contemporary knowledge work, wherein individuals must assume the role of their own chief executive officers due to the absence of corporate oversight in career trajectories. Drucker posits that success is contingent upon self-knowledge, necessitating introspective evaluations of one's strengths, learning modalities, values, and optimal work environments. He advocates for the implementation of feedback analysis to accurately discern personal strengths, urging individuals to concentrate on and enhance these attributes while eschewing areas of incompetence. The discourse further addresses the recognition of diverse performance styles and learning preferences, emphasizing that self-awareness in these dimensions is crucial for achieving results in collaborative settings. Additionally, Drucker underscores the importance of aligning one’s values with those of the organizations they engage with to mitigate frustration and enhance effecti

In [23]:
#Evaluating the new summary with the same function
enhanced_evaluation = evaluate_summary(
    document_text=document_text,
    summary_text=enhanced_summary_text,
    tone=tone,
    eval_model=eval_model
)

print(enhanced_evaluation.model_dump())


Output()

Output()

Output()

Output()

{'SummarizationScore': 1.0, 'SummarizationReason': 'The score is 1.00 because the summary accurately reflects the original text without any contradictions or extra information, demonstrating a perfect alignment with the source material.', 'CoherenceScore': 0.8705785021648482, 'CoherenceReason': "The response effectively summarizes the key themes of Drucker's work on self-management, following a logical order and connecting ideas clearly. It highlights the importance of self-knowledge, feedback analysis, and aligning personal values with organizational values, which are central to the original text. The writing is clear and easy to understand, avoiding contradictions. However, it could benefit from a more explicit mention of the specific questions Drucker suggests individuals ask themselves, which would enhance the overall structure and organization of the summary.", 'TonalityScore': 0.2686854909383182, 'TonalityReason': 'The summary lacks the requested legalese tone, instead presenting

In [26]:
#Comparison

comparison = {
    "OriginalSummary": summary_text,
    "EnhancedSummary": enhanced_summary_text,
    "InitialEvaluation": initial_evaluation.model_dump(),
    "EnhancedEvaluation": enhanced_evaluation.model_dump(),
}

comparison

{'OriginalSummary': "This text elucidates the necessity of self-management in contemporary knowledge work. As companies no longer oversee their employees' career paths, individuals must become their own chief executive officers. Drucker posits that success hinges on self-knowledge, prompting professionals to introspectively evaluate their strengths, learning modalities, values, and optimal work environments. He advocates for feedback analysis to accurately discern personal strengths, encouraging individuals to concentrate on and amplify these attributes while avoiding areas of incompetence. The discourse extends to recognizing diverse performance styles and learning preferences, with Drucker underscoring that self-awareness in these dimensions is crucial for achieving results in collaborative settings. Furthermore, he navigates the significance of aligning one’s values with those of the organizations they engage with to circumvent frustration and enhance effectiveness. The text conclud

In [27]:
#Cleaner score table
print("Initial Summarization Score:", initial_evaluation.SummarizationScore)
print("Enhanced Summarization Score:", enhanced_evaluation.SummarizationScore)
print()

print("Initial Coherence Score:", initial_evaluation.CoherenceScore)
print("Enhanced Coherence Score:", enhanced_evaluation.CoherenceScore)
print()

print("Initial Tonality Score:", initial_evaluation.TonalityScore)
print("Enhanced Tonality Score:", enhanced_evaluation.TonalityScore)
print()

print("Initial Safety Score:", initial_evaluation.SafetyScore)
print("Enhanced Safety Score:", enhanced_evaluation.SafetyScore)

Initial Summarization Score: 0.8571428571428571
Enhanced Summarization Score: 1.0

Initial Coherence Score: 0.8731058572770513
Enhanced Coherence Score: 0.8705785021648482

Initial Tonality Score: 0.2536112430024171
Enhanced Tonality Score: 0.2686854909383182

Initial Safety Score: 0.930084315840706
Enhanced Safety Score: 0.950387348087238


Please, do not forget to add your comments.

### Enhancement results (My comments - E. Fantinatti)

To improve the original summary, I created a second prompt that used three inputs:
1. the original article text,
2. the first generated summary,
3. the evaluation scores and reasons from DeepEval.

The goal was to create a self-correction loop in which the model revised the summary based on evaluator feedback. I then re-ran the same evaluation function on the revised summary and compared the results.

In my case, the enhanced summary [improved / did not improve] the evaluation scores. This happened because the second prompt explicitly instructed the model to correct issues identified by the evaluator, especially around factual faithfulness and clarity.

These controls help, but they are not sufficient on their own. The evaluator is still another model-based judge, so it can be imperfect and somewhat variable. The quality of the feedback also depends on the prompt, the selected metrics, and the source text. For that reason, automated evaluation is useful for iterative improvement, but important outputs may still benefit from human review.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
